## Perceptron

> Yeh notebook [AI for Beginners Curricula](http://github.com/microsoft/ai-for-beginners) ka hissa hai. Poora learning material dekhne ke liye repository par jaayein.

Jaisa ki humne discuss kiya, Perceptron aapko **binary classification problem** solve karne deta hai, yaani input examples ko do classes mein classify karna - hum unhe **positive** aur **negative** keh sakte hain.

Sabse pehle, kuch zaroori libraries import karte hain.

In [ ]:
import pylab
from matplotlib import gridspec
from sklearn.datasets import make_classification
import numpy as np
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets
import pickle
import os
import gzip

# pick the seed for reproducability - change it to explore the effects of random variations
np.random.seed(1)
import random

## Toy Problem

Shuruaat ke liye, chalo ek toy problem se start karte hain, jahan hamare paas do input features hain. Jaise medicine mein hum tumor ko uske size aur age ke basis par benign (saumya) aur malignant (ghaatak) mein classify karna chah sakte hain.

Hum SciKit Learn library ke `make_classification` function se ek random classification dataset banayenge:

In [ ]:
n = 50
X, Y = make_classification(n_samples = n, n_features=2,
                           n_redundant=0, n_informative=2, flip_y=0)
Y = Y*2-1 # convert initial 0/1 values into -1/1
X = X.astype(np.float32); Y = Y.astype(np.int32) # features - float, label - int

# Split the dataset into training and test
train_x, test_x = np.split(X, [ n*8//10])
train_labels, test_labels = np.split(Y, [n*8//10])
print("Features:\n",train_x[0:4])
print("Labels:\n",train_labels[0:4])

Chalo dataset ko bhi plot karte hain:

In [ ]:
def plot_dataset(suptitle, features, labels):
    # prepare the plot
    fig, ax = pylab.subplots(1, 1)
    #pylab.subplots_adjust(bottom=0.2, wspace=0.4)
    fig.suptitle(suptitle, fontsize = 16)
    ax.set_xlabel('$x_i[0]$ -- (feature 1)')
    ax.set_ylabel('$x_i[1]$ -- (feature 2)')

    colors = ['r' if l>0 else 'b' for l in labels]
    ax.scatter(features[:, 0], features[:, 1], marker='o', c=colors, s=100, alpha = 0.5)
    fig.show()

plot_dataset('Training data', train_x, train_labels)

## Perceptron

Kyunki Perceptron ek binary classifier hai, har input vector $x$ ke liye hamare perceptron ka output +1 ya -1 hoga, class ke hisaab se. Output is formula se calculate hoga:

$$y(\mathbf{x}) = f(\mathbf{w}^{\mathrm{T}}\mathbf{x})$$

jahan $\mathbf{w}$ ek weight vector hai, $f$ ek step activation function hai:
$$
f(x) = \begin{cases}
         +1 & x \geq 0 \\
         -1 & x < 0
       \end{cases} \\
$$

Lekin, ek general linear model mein bias bhi hona chahiye, yaani ideal roop mein hume $y$ ko $y=f(\mathbf{w}^{\mathrm{T}}\mathbf{x}+\mathbf{b})$ ke roop mein calculate karna chahiye. Apne model ko simple banane ke liye, hum is bias term ko hata sakte hain, apne input features mein ek aur dimension add karke, jo hamesha 1 ke barabar hota hai:

In [ ]:
pos_examples = np.array([ [t[0], t[1], 1] for i,t in enumerate(train_x) 
                          if train_labels[i]>0])
neg_examples = np.array([ [t[0], t[1], 1] for i,t in enumerate(train_x) 
                          if train_labels[i]<0])
print(pos_examples[0:3])

## Training Algorithm

Perceptron ko train karne ke liye, hume aise weights $\mathbf{w}$ dhoondhne honge jo error ko minimize karein. Error ko **perceptron criterion** use karke define kiya gaya hai:

$$E(\mathbf{w}) = -\sum_{n \in \mathcal{M}}\mathbf{w}^{\mathrm{T}}\mathbf{x}_{n}t_{n}$$

  * $t_{n} \in \{-1, +1\}$ negative aur positive training samples ke liye, respectively
  * $\mathcal{M}$ - galat classify hue examples ka set

Hum **gradient descent** ka process use karenge. Kuch initial random weights $\mathbf{w}^{(0)}$ se shuru karke, hum training ke har step mein $E$ ke gradient ka use karke weights ko adjust karenge:

$$\mathbf{w}^{\tau + 1}=\mathbf{w}^{\tau} - \eta \nabla E(\mathbf{w}) = \mathbf{w}^{\tau} + \eta\sum_{n \in \mathcal{M}}\mathbf{x}_{n} t_{n}$$

jahan $\eta$ ek **learning rate** hai, aur $\tau\in\mathbb{N}$ - iterations ki number.

Chalo is algorithm ko Python mein define karte hain:

In [ ]:
def train(positive_examples, negative_examples, num_iterations = 100, learning_rate = 0.01):
    num_dims = positive_examples.shape[1]
    
    # Initialize weights. 
    # We initialize with 0 for simplicity, but random initialization is also a good idea
    weights = np.zeros((num_dims,1)) 
    
    pos_count = positive_examples.shape[0]
    neg_count = negative_examples.shape[0]
    
    report_frequency = 10
    
    for i in range(num_iterations):
        # Pick one positive and one negative example
        pos = random.choice(positive_examples)
        neg = random.choice(negative_examples)

        z = np.dot(pos, weights)   
        if z < 0: # positive example was classified as negative
            weights = weights + learning_rate * pos.reshape(weights.shape)

        z  = np.dot(neg, weights)
        if z >= 0: # negative example was classified as positive
            weights = weights - learning_rate * neg.reshape(weights.shape)
            
        # Periodically, print out the current accuracy on all examples 
        if i % report_frequency == 0:             
            pos_out = np.dot(positive_examples, weights)
            neg_out = np.dot(negative_examples, weights)        
            pos_correct = (pos_out >= 0).sum() / float(pos_count)
            neg_correct = (neg_out < 0).sum() / float(neg_count)
            print("Iteration={}, pos correct={}, neg correct={}".format(i,pos_correct,neg_correct))

    return weights

**Learning Rate par Note**: `learning_rate` parameter (default `0.01`) control karta hai ki har training step ke dauraan weights ko kitna adjust kiya jaaye. Yeh gradient descent update formula ko implement karta hai:

$$\mathbf{w}^{\tau + 1}=\mathbf{w}^{\tau} + \eta \mathbf{x}_{n} t_{n}$$

- Bada learning rate (jaise, `1.0`) perceptron ko tezi se seekhne mein help karta hai lekin woh ideal solution ko paar (overshoot) kar sakta hai
- Chhota learning rate (jaise, `0.001`) dheere seekhta hai lekin zyada accurately converge kar sakta hai
- Aap ise try karne ke liye call kar sakte hain: `train(pos_examples, neg_examples, learning_rate=0.1)`

Ab chalo apne dataset par training shuru karte hain:

In [ ]:
wts = train(pos_examples,neg_examples)
print(wts.transpose())

Jaisa ki aap dekh sakte hain, initial accuracy lagbhag 50% hai, lekin yeh jaldi hi badhkar 90% ke aas-paas high values tak pahunch jaati hai.

Chalo dekhte hain ki classes kaise alag hoti hain. Hamara classification function $\mathbf{w}^Tx$ jaisa dikhta hai, aur yeh ek class ke liye 0 se zyada hai, aur doosri ke liye 0 se kam. Is tarah, class separation line $\mathbf{w}^Tx = 0$ se define hoti hai. Kyunki hamare paas sirf do dimensions $x_0$ aur $x_1$ hain, line ki equation $w_0x_0+w_1x_1+w_2 = 0$ hogi (yaad rakhiye humne explicitly ek extra dimension $x_2=1$ define kiya tha). Chalo is line ko plot karte hain:

In [ ]:
def plot_boundary(positive_examples, negative_examples, weights):
    if np.isclose(weights[1], 0):
        if np.isclose(weights[0], 0):
            x = y = np.array([-6, 6], dtype = 'float32')
        else:
            y = np.array([-6, 6], dtype='float32')
            x = -(weights[1] * y + weights[2])/weights[0]
    else:
        x = np.array([-6, 6], dtype='float32')
        y = -(weights[0] * x + weights[2])/weights[1]

    pylab.xlim(-6, 6)
    pylab.ylim(-6, 6)                      
    pylab.plot(positive_examples[:,0], positive_examples[:,1], 'bo')
    pylab.plot(negative_examples[:,0], negative_examples[:,1], 'ro')
    pylab.plot(x, y, 'g', linewidth=2.0)
    pylab.show()

In [ ]:
plot_boundary(pos_examples,neg_examples,wts)

## Learning Rates ke saath Experiment karna

Ab dekhte hain ki alag-alag learning rates training process ko kaise affect karte hain. Learning rate gradient descent mein step size ko control karta hai - yeh ek important hyperparameter hai jo convergence ki speed aur stability dono ko affect karta hai.

In [ ]:
# Compare different learning rates
learning_rates = [0.001, 0.01, 0.1, 1.0]
fig, axes = pylab.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Effect of Different Learning Rates', fontsize=16)

for idx, lr in enumerate(learning_rates):
    ax = axes[idx // 2, idx % 2]
    
    # Train with this learning rate
    weights_lr = train(pos_examples, neg_examples, num_iterations=100, learning_rate=lr)
    
    # Plot decision boundary
    if np.isclose(weights_lr[1], 0):
        if np.isclose(weights_lr[0], 0):
            x = y = np.array([-6, 6], dtype='float32')
        else:
            y = np.array([-6, 6], dtype='float32')
            x = -(weights_lr[1] * y + weights_lr[2])/weights_lr[0]
    else:
        x = np.array([-6, 6], dtype='float32')
        y = -(weights_lr[0] * x + weights_lr[2])/weights_lr[1]
    
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.plot(pos_examples[:, 0], pos_examples[:, 1], 'bo', label='Positive', alpha=0.7)
    ax.plot(neg_examples[:, 0], neg_examples[:, 1], 'ro', label='Negative', alpha=0.7)
    ax.plot(x, y, 'g-', linewidth=2)
    ax.set_title(f'Learning Rate = {lr}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend()
    ax.grid(True, alpha=0.3)

pylab.tight_layout()
pylab.show()


### Interactive Learning Rate Experiment

Neeche diye gaye slider se alag-alag learning rates ke saath interactively experiment karein aur dekhein ki woh decision boundary ko kaise affect karte hain:

In [ ]:
def train_and_plot_with_lr(learning_rate=0.01):
    """Train perceptron with specified learning rate and plot results"""
    weights_lr = train(pos_examples, neg_examples, num_iterations=100, learning_rate=learning_rate)
    
    fig, (ax1, ax2) = pylab.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Decision boundary
    if np.isclose(weights_lr[1], 0):
        if np.isclose(weights_lr[0], 0):
            x = y = np.array([-6, 6], dtype='float32')
        else:
            y = np.array([-6, 6], dtype='float32')
            x = -(weights_lr[1] * y + weights_lr[2])/weights_lr[0]
    else:
        x = np.array([-6, 6], dtype='float32')
        y = -(weights_lr[0] * x + weights_lr[2])/weights_lr[1]
    
    ax1.set_xlim(-6, 6)
    ax1.set_ylim(-6, 6)
    ax1.plot(pos_examples[:, 0], pos_examples[:, 1], 'bo', label='Positive', s=100, alpha=0.6)
    ax1.plot(neg_examples[:, 0], neg_examples[:, 1], 'ro', label='Negative', s=100, alpha=0.6)
    ax1.plot(x, y, 'g-', linewidth=3, label='Decision Boundary')
    ax1.set_title(f'Decision Boundary (lr={learning_rate})', fontsize=14)
    ax1.set_xlabel('Feature 1')
    ax1.set_ylabel('Feature 2')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Weight values
    ax2.bar(['w0', 'w1', 'bias'], weights_lr.flatten(), color=['blue', 'green', 'red'], alpha=0.7)
    ax2.set_title('Final Weight Values', fontsize=14)
    ax2.set_ylabel('Weight Value')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    pylab.tight_layout()
    pylab.show()
    
    print(f"Final weights: {weights_lr.flatten()}")

# Create interactive widget
interact(train_and_plot_with_lr, 
         learning_rate=widgets.FloatSlider(value=0.01, min=0.001, max=1.0, step=0.001, 
                                          description='Learning Rate:', continuous_update=False))


## Test Dataset par Evaluate karein

Shuruaat mein, humne kuch data test dataset ke liye alag rakha tha. Chalo dekhte hain ki hamara classifier is test dataset par kitna accurate hai. Iske liye, hum test dataset ko ek extra dimension ke saath extend karte hain, weights matrix se multiply karte hain, aur ensure karte hain ki jo value mile woh label (+1 ya -1) ke same sign ki ho. Phir hum saari boolean values ko add karte hain aur test samples ki length se divide karte hain, taaki accuracy mil sake:

In [ ]:
def accuracy(weights, test_x, test_labels):
    res = np.dot(np.c_[test_x,np.ones(len(test_x))],weights)
    return (res.reshape(test_labels.shape)*test_labels>=0).sum()/float(len(test_labels))

accuracy(wts, test_x, test_labels)

## Training Process ko Observe karna

Humne pehle dekha ki training ke dauraan accuracy kaise badalti hai. Yeh dekhna accha hoga ki training ke dauraan separation line kaise behave karti hai. Neeche diya gaya code sab kuch ek graph par dikhayega, aur aap slider ko khiska kar training process ke dauraan "time travel" kar paayenge.

In [ ]:
def train_graph(positive_examples, negative_examples, num_iterations = 100, learning_rate = 0.01):
    num_dims = positive_examples.shape[1]
    weights = np.zeros((num_dims,1)) # initialize weights
    
    pos_count = positive_examples.shape[0]
    neg_count = negative_examples.shape[0]
    
    report_frequency = 15;
    snapshots = []
    
    for i in range(num_iterations):
        pos = random.choice(positive_examples)
        neg = random.choice(negative_examples)

        z = np.dot(pos, weights)   
        if z < 0:
            weights = weights + learning_rate * pos.reshape(weights.shape)

        z  = np.dot(neg, weights)
        if z >= 0:
            weights = weights - learning_rate * neg.reshape(weights.shape)
            
        if i % report_frequency == 0:             
            pos_out = np.dot(positive_examples, weights)
            neg_out = np.dot(negative_examples, weights)        
            pos_correct = (pos_out >= 0).sum() / float(pos_count)
            neg_correct = (neg_out < 0).sum() / float(neg_count)
            snapshots.append([np.copy(weights).flatten(), (pos_correct+neg_correct)/2.0])

    return np.array(snapshots, dtype=object)

snapshots = train_graph(pos_examples,neg_examples)

def plotit(pos_examples,neg_examples,snapshots,step):
    fig = pylab.figure(figsize=(10,4))
    fig.add_subplot(1, 2, 1)
    plot_boundary(pos_examples, neg_examples, snapshots[step][0])
    fig.add_subplot(1, 2, 2)
    pylab.plot(np.arange(len(snapshots[:,1])), snapshots[:,1])
    pylab.ylabel('Accuracy')
    pylab.xlabel('Iteration')
    pylab.plot(step, snapshots[step,1], "bo")
    pylab.show()
def pl1(step): plotit(pos_examples,neg_examples,snapshots,step)

In [ ]:
interact(pl1, step=widgets.IntSlider(value=0, min=0, max=len(snapshots)-1))

## Perceptron ki Limitations

Jaisa ki aapne upar dekha, perceptron ek **linear classifier** hai. Yeh do classes ke beech fark kar sakta hai agar woh **linearly separable** hon, yaani unhe ek seedhi line se alag kiya ja sake. Warna, perceptron ka training process converge nahi karega.

Sabse clear example, jise perceptron solve nahi kar sakta, woh hai tathaakathit **XOR problem**. Hum chahte hain ki hamara perceptron XOR boolean function seekhe, jiski truth table yeh hai:

|   | 0 | 1 |
|---|---|---|
| 0 | 0 | 1 |
| 1 | 1 | 0 |

Chalo try karte hain! Hum saare positive aur negative training samples manually bharenge, aur phir upar define kiye gaye apne train function ko call karenge:

In [ ]:
pos_examples_xor = np.array([[1,0,1],[0,1,1]])
neg_examples_xor = np.array([[1,1,1],[0,0,1]])

snapshots_xor = train_graph(pos_examples_xor,neg_examples_xor,1000)
def pl2(step): plotit(pos_examples_xor,neg_examples_xor,snapshots_xor,step)

In [ ]:
interact(pl2, step=widgets.IntSlider(value=0, min=0, max=len(snapshots)-1))

Jaisa ki upar ke graph se dekha ja sakta hai, accuracy kabhi 75% se upar nahi jaati, kyunki saare possible examples ko sahi tarike se paane ke liye ek seedhi line kheenchna impossible hai.

XOR problem perceptron ki limitations ka ek classical example hai, aur ise 1969 mein Marvin Minsky aur Seymour Papert ne apni book [Perceptrons](https://en.wikipedia.org/wiki/Perceptrons_(book)) mein bataya tha. Is observation ne lagbhag 10 saal tak neural network ke field mein research ko limit kar diya, haalaanki - aur hum ise course ke agle hisse mein dekhenge - multi-layered perceptrons aisi problems ko solve karne mein poori tarah capable hain.

## Complex Example - MNIST

Haalaanki perceptron XOR problem solve nahi kar sakta, yeh kai zyada complex problems solve kar sakta hai, jaise handwritten letters ko recognize karna.

Machine learning seekhte waqt aksar use hone waala ek dataset [MNIST](https://en.wikipedia.org/wiki/MNIST_database) kehlata hai. Ise Modified National Institute of Standards and Technology ne banaya hai, aur isme 60000 handwritten digits ka training set hai, jo lagbhag 250 students aur institute ke employees se collect kiye gaye hain. Iske alawa, 10000 digits ka ek test dataset bhi hai, jo alag-alag logon se collect kiya gaya hai.

Saare digits 28x28 pixels ki grayscale images se represent kiye gaye hain.

> MNIST dataset [Kaggle](https://www.kaggle.com/c/digit-recognizer) par ek training competition ke roop mein available hai, jo machine learning competitions host karta hai. Ek baar jab aap MNIST digits ko classify karna seekh jaayein, toh aap apna solution Kaggle par submit karke dekh sakte hain ki woh doosre participants ke beech kaise rate hota hai.

Hum MNIST dataset load karke shuru karte hain:

In [ ]:
# If you are not running this notebook from a cloned repository, you may need to grab the binary dataset file first
# !wget https://github.com/microsoft/AI-For-Beginners/raw/main/data/mnist.pkl.gz?raw=true
# In this case correct the link to the dataset below as well.

with gzip.open('../../../data/mnist.pkl.gz', 'rb') as mnist_pickle:
    MNIST = pickle.load(mnist_pickle, encoding='latin1')

Ab dataset ko plot karte hain:

In [ ]:
print(MNIST['Train']['Features'][0][130:180])
print(MNIST['Train']['Labels'][0])
features = MNIST['Train']['Features'].astype(np.float32) / 256.0
labels = MNIST['Train']['Labels']
fig = pylab.figure(figsize=(10,5))
for i in range(10):
    ax = fig.add_subplot(1,10,i+1)
    pylab.imshow(features[i].reshape(28,28))
pylab.show()

Kyunki perceptron ek binary classifier hai, hum apni problem ko sirf do digits recognize karne tak limit karenge. Neeche diya gaya function do diye gaye digits ke saath positive aur negative sample arrays bhar dega (aur clarity ke liye un digits ke samples bhi dikhayega).

In [ ]:
def set_mnist_pos_neg(positive_label, negative_label):
    positive_indices = [i for i, j in enumerate(MNIST['Train']['Labels']) 
                          if j == positive_label]
    negative_indices = [i for i, j in enumerate(MNIST['Train']['Labels']) 
                          if j == negative_label]

    positive_images = MNIST['Train']['Features'][positive_indices]
    negative_images = MNIST['Train']['Features'][negative_indices]

    fig = pylab.figure()
    ax = fig.add_subplot(1, 2, 1)
    pylab.imshow(positive_images[0].reshape(28,28), cmap='gray', interpolation='nearest')
    ax.set_xticks([])
    ax.set_yticks([])
    ax = fig.add_subplot(1, 2, 2)
    pylab.imshow(negative_images[0].reshape(28,28), cmap='gray', interpolation='nearest')
    ax.set_xticks([])
    ax.set_yticks([])
    pylab.show()
    
    return positive_images, negative_images

Hum 0 aur 1 ke beech classify karne ki koshish se shuru karenge:

In [ ]:
pos1,neg1 = set_mnist_pos_neg(1,0)

In [ ]:
def plotit2(snapshots_mn,step):
    fig = pylab.figure(figsize=(10,4))
    ax = fig.add_subplot(1, 2, 1)
    pylab.imshow(snapshots_mn[step][0].reshape(28, 28), interpolation='nearest')
    ax.set_xticks([])
    ax.set_yticks([])
    pylab.colorbar()
    ax = fig.add_subplot(1, 2, 2)
    ax.set_ylim([0,1])
    pylab.plot(np.arange(len(snapshots_mn[:,1])), snapshots_mn[:,1])
    pylab.plot(step, snapshots_mn[step,1], "bo")
    pylab.show()
def pl3(step): plotit2(snapshots_mn,step)
def pl4(step): plotit2(snapshots_mn2,step)    

In [ ]:
snapshots_mn = train_graph(pos1,neg1,1000)    
interact(pl3, step=widgets.IntSlider(value=0, min=0, max=len(snapshots_mn) - 1))

Dhyan dijiye ki accuracy bahut tezi se lagbhag 100% tak pahunch jaati hai.

Please, slider ko training ke end ki taraf kisi position par le jaayein, aur left side par plot kiye gaye weights matrix ko dekhein. Yeh matrix aapko samajhne mein help karega ki perceptron actually kaise kaam karta hai. Aap area ke beech mein high weight values dekh sakte hain, jo un pixels se match karti hain jo aam taur par digit 1 ke liye present hote hain, aur edges par low negative values, jahan digit 0 ke parts hote hain. Toh, agar perceptron ko diya gaya digit sach mein 1 hai, toh uska middle part high values se multiply hoga, jisse positive result aayega. Iske ulat, jab perceptron 0 dekhta hai, toh corresponding pixels negative numbers se multiply honge.

> Aap dekh sakte hain ki agar hum apne perceptron ko thoda horizontally khiska hua digit 1 dein, taaki uske pixels us jagah aa jaayein jahan 0 ke vertical parts hote hain, toh hume galat result mil sakta hai. Kyunki hamare MNIST dataset ki nature aisi hai ki saare digits centered aur sahi tarike se positioned hote hain, aur perceptron isi par depend karta hai digits ke beech fark karne ke liye.

Ab chalo alag-alag digits try karte hain:

In [ ]:
pos2,neg2 = set_mnist_pos_neg(2,5)

In [ ]:
snapshots_mn2 = train_graph(pos2,neg2,1000)
interact(pl4, step=widgets.IntSlider(value=0, min=0, max=len(snapshots_mn2) - 1))

## Discussion

Kisi wajah se, 2 aur 5 ko aasaani se alag karna mushkil hota hai. Bhale hi hume relatively high accuracy (85% se zyada) mil rahi hai, hum clearly dekh sakte hain ki kisi point par perceptron seekhna band kar deta hai.

Yeh samajhne ke liye ki aisa kyun hota hai, hum [Principal Component Analysis](https://en.wikipedia.org/wiki/Principal_component_analysis) (PCA) use kar sakte hain. Yeh ek machine learning technique hai jo input dataset ki dimensionality ko kam karne ke liye use hoti hai, taaki classes ke beech best separation mil sake.

Hamare case mein, ek input image mein 784 pixels (input features) hote hain, aur hum PCA use karke parameters ki number ko sirf 2 tak kam karna chahte hain, taaki hum unhe graph par plot kar sakein. Yeh do parameters original features ka linear combination honge, aur hum is process ko aise dekh sakte hain jaise hum apne original 784-dimensional space ko "ghuma" rahe hain aur use 2D-space par projection ke roop mein dekh rahe hain, jab tak hume classes ko alag karne ke liye sabse accha view na mil jaaye.

In [ ]:
from sklearn.decomposition import PCA

def pca_analysis(positive_label, negative_label):
    positive_images, negative_images = set_mnist_pos_neg(positive_label, negative_label)
    M = np.append(positive_images, negative_images, 0)

    mypca = PCA(n_components=2)
    mypca.fit(M)
    
    pos_points = mypca.transform(positive_images[:200])
    neg_points = mypca.transform(negative_images[:200])

    pylab.plot(pos_points[:,0], pos_points[:,1], 'bo')
    pylab.plot(neg_points[:,0], neg_points[:,1], 'ro')

In [ ]:
pca_analysis(1,0)

In [ ]:
pca_analysis(2,5)

Jaisa ki aap dekh sakte hain, 0 aur 1 ko ek seedhi line se clearly alag kiya ja sakta hai. Yeh dikhata hai ki original 784-dimensional space mein digits se related points bhi linearly separable hain. Lekin 2 aur 5 ke case mein, hume aisa accha projection nahi mil sakta jo digits ko clearly alag kar sake, aur isliye kuch misclassification ke cases saamne aate hain.

> Is course mein aage hum seekhenge ki Neural Networks use karke non-linear classifiers kaise banaye jaate hain, aur un problems se kaise nipta jaaye jahan digits sahi tarike se aligned nahi hote. Bahut jaldi hum MNIST digits ki classification mein 99% se zyada accuracy hasil karenge, jabki unhe 10 alag classes mein classify karenge.

## Main Takeaways

 * Humne sabse simple Neural Network architecture - single-layer perceptron - ke baare mein seekha.
 * Humne perceptron ko "haath se" implement kiya, simple training procedure use karke jo gradient descent par based hai.
 * Simplicity ke bawajood, single-layer perceptron handwritten digits recognize karne jaisi complex problems solve kar sakta hai.
 * Single-layer perceptron ek linear classifier hai, aur isliye yeh logistic regression jaisi hi classification capability deta hai.
 * Sample space mein, perceptron hyperplane use karke input data ki do classes ko alag kar sakta hai.

## Credits

Yeh notebook [AI for Beginners Curricula](http://github.com/microsoft/ai-for-beginners) ka hissa hai, aur ise [Dmitry Soshnikov](http://soshnikov.com) ne taiyaar kiya hai. Yeh Microsoft Research Cambridge ke Neural Network Workshop se inspired hai. Kuch code aur illustrative material [Katja Hoffmann](https://www.microsoft.com/en-us/research/people/kahofman/), [Matthew Johnson](https://www.microsoft.com/en-us/research/people/matjoh/) aur [Ryoto Tomioka](https://www.microsoft.com/en-us/research/people/ryoto/) ki presentations se liya gaya hai, aur [NeuroWorkshop](http://github.com/shwars/NeuroWorkshop) repository se bhi.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->